# 02 - Derivations

## Setup

In [12]:
import pandas as pd
import numpy as np
from numpy.linalg import norm

OUTPUT_DIR = 'tables'

LIB = pd.read_csv('tables/LIB.csv')
CORPUS = pd.read_csv('tables/CORPUS.csv')
VOCAB = pd.read_csv('tables/VOCAB.csv')

In [13]:
OHCO = ['film_name', 'chunk_num', 'sent_num', 'token_num']
bags = dict(
    SENTS = OHCO[:3],
    CHUNKS = OHCO[:2],
    FILMS = OHCO[:1]
)

In [14]:
CORPUS = CORPUS.set_index(OHCO).dropna()
CORPUS  

token_str  term_str  speaker  \
film_name         chunk_num sent_num token_num                                
castle_in_the_sky 0         0        0                Ah        ah      MEN   
                  1         0        0               Wah       wah  CREWMAN   
                            1        0                It        it  CREWMAN   
                                     1                's         s  CREWMAN   
                                     2                 a         a  CREWMAN   
...                                                  ...       ...      ...   
princess_kaguya   1221      1        0          Princess  princess  UNKNOWN   
                  1222      0        0          Princess  princess  UNKNOWN   
                            1        0          Princess  princess  UNKNOWN   
                  1223      0        0           Forgive   forgive  UNKNOWN   
                                     1                me        me  UNKNOWN   

                                                pos pos_group  
film_name         chunk_num sent_num token_num                 
castle_in_the_sky 0         0        0           NN         n  
                  1         0        0           NN         n  
                            1        0          PRP         x  
                                     1          VBZ         v  
                                     2           DT         x  
...                                             ...       ...  
princess_kaguya   1221      1        0           NN         n  
                  1222      0        0           NN         n  
                            1        0           NN         n  
                  1223      0        0           CD         x  
                                     1          PRP         x  

[95992 rows x 5 columns]

## BOW

In [15]:
def get_BOW(token_df, bag):
    BOW = token_df.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n')
    return(BOW)

BOW = get_BOW(CORPUS, 'FILMS')
BOW

n
film_name         term_str      
castle_in_the_sky 10           3
                  2            1
                  3            1
                  40           2
                  50           1
...                           ..
the_wind_rises    your        51
                  yours        1
                  yourself     6
                  zauberberg   1
                  zero         1

[16992 rows x 1 columns]

## DTM

In [16]:
DTM = BOW.n.unstack(fill_value=0)
DTM

term_str,0,05,1,10,100,1000,1000000,105,12,120,...,yyes,yyou,zaoh,zauberberg,zeniba,zero,zo,zohsui,zone,zosui
film_name,,,,,,,,,,,,,,,,,,,,,
castle_in_the_sky,0,0,0,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
grave_of_the_fireflies,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,2,1,0,2
howls_moving_castle,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
kikis_delivery_service,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
my_neighbor_totoro,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nausicaa,0,0,33,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
only_yesterday,0,0,4,0,2,0,0,0,0,0,...,0,0,6,0,0,0,0,0,0,0
pom_poko,0,0,1,1,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,2,0
ponyo,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## DFIDF + Top 20

In [17]:
VOCAB = VOCAB.set_index('term_str')

DF = DTM.astype('bool').sum()
N = DTM.shape[0]
IDF = np.log2(N / DF) # Standard IDF method

VOCAB['df'] = DF
VOCAB['idf'] = IDF
VOCAB['dfidf'] = (DF / N) * VOCAB['idf'] # = mean boolean TFIDF

VOCAB.sort_values('dfidf', ascending=False).head(20)

,n,p,i,n_chars,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,stem_porter,stem_snowball,stem_lancaster,df,idf,dfidf
term_str,,,,,,,,,,,,,,,,,
sooner,6,0.000063,13.965724,6,NN,n,3,"{'j', 'r', 'n'}",4,"{'RBR', 'RB', 'NN', 'JJR'}",False,sooner,sooner,soon,5,1.485427,0.53051
attacked,9,0.000094,13.380762,8,VBD,v,1,{'v'},2,"{'VBN', 'VBD'}",False,attack,attack,attack,5,1.485427,0.53051
received,7,0.000073,13.743332,8,VBD,v,1,{'v'},2,"{'VBN', 'VBD'}",False,receiv,receiv,receiv,5,1.485427,0.53051
knock,5,0.000052,14.228759,5,VB,v,2,"{'n', 'v'}",2,"{'VB', 'NN'}",False,knock,knock,knock,5,1.485427,0.53051
treat,5,0.000052,14.228759,5,VB,v,2,"{'n', 'v'}",3,"{'VB', 'VBP', 'NNP'}",False,treat,treat,tre,5,1.485427,0.53051
escape,8,0.000083,13.550687,6,VB,v,2,"{'n', 'v'}",3,"{'VB', 'VBP', 'NN'}",False,escap,escap,escap,5,1.485427,0.53051
calling,8,0.000083,13.550687,7,VBG,v,1,{'v'},1,{'VBG'},False,call,call,cal,5,1.485427,0.53051
hour,7,0.000073,13.743332,4,NN,n,1,{'n'},1,{'NN'},False,hour,hour,hour,5,1.485427,0.53051
allow,6,0.000063,13.965724,5,VB,v,2,"{'n', 'v'}",2,"{'VB', 'NN'}",False,allow,allow,allow,5,1.485427,0.53051


## TFIDF

In [18]:
def get_TFIDF(BOW, tf_method, idf_method):
    # Document Term Count Matrix 
    DTCM = BOW.n.unstack(fill_value=0)

    # Compute TF
    if tf_method == 'sum':
        TF = DTCM.T / DTCM.T.sum()
        
    elif tf_method == 'smooth':
        TF = (DTCM.T / DTCM.T.sum()) + 1
    
    elif tf_method == 'max':
        TF = DTCM.T / DTCM.T.max()
        
    elif tf_method == 'log':
        TF = np.log2(1 + DTCM.T)
        
    elif tf_method == 'raw':
        TF = DTCM.T
        
    elif tf_method == 'double_norm':
        TF = DTCM.T / DTCM.T.max()
        
    elif tf_method == 'binary':
        TF = DTCM.T.astype('bool').astype('int')

    TF = TF.T

    # Compute DF
    DF = DTCM.astype('bool').sum() 

    # Compute IDF
    N = DTCM.shape[0]
    
    if idf_method == 'standard':
        IDF = np.log2(N / DF)

    elif idf_method == 'max':
        IDF = np.log2(DF.max() / DF) 
    
    elif idf_method == 'plus':
        IDF = np.log2(N / DF) + 1
    
    elif idf_method == 'smooth': # what SciKit Learn uses
        IDF = np.log2((1 + N) / (1 + DF)) + 1

    # Compute TFIDF
    TFIDF = TF * IDF

    return(TFIDF)

In [19]:
TFIDF = get_TFIDF(BOW, tf_method = 'sum', idf_method = 'standard')
TFIDF

term_str,0,05,1,10,100,1000,1000000,105,12,120,...,yyes,yyou,zaoh,zauberberg,zeniba,zero,zo,zohsui,zone,zosui
film_name,,,,,,,,,,,,,,,,,,,,,
castle_in_the_sky,0.00000,0.00000,0.000000,0.000680,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
grave_of_the_fireflies,0.00000,0.00000,0.000000,0.000349,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001787,0.000894,0.000000,0.001787
howls_moving_castle,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000439,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
kikis_delivery_service,0.00000,0.00000,0.000000,0.000215,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
my_neighbor_totoro,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
nausicaa,0.00000,0.00000,0.008324,0.000000,0.000000,0.000392,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
only_yesterday,0.00000,0.00000,0.000988,0.000000,0.001041,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.003122,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pom_poko,0.00000,0.00000,0.000145,0.000119,0.000000,0.000225,0.000305,0.000305,0.000225,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000609,0.000000
ponyo,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Reduced and Normalized TFIDF_L2

In [20]:
TFIDF_L2 = TFIDF.apply(lambda x: x / norm(x), 1) # Pythagorean, aka Euclidean
TFIDF_L2

term_str,0,05,1,10,100,1000,1000000,105,12,120,...,yyes,yyou,zaoh,zauberberg,zeniba,zero,zo,zohsui,zone,zosui
film_name,,,,,,,,,,,,,,,,,,,,,
castle_in_the_sky,0.000000,0.000000,0.000000,0.010320,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000
grave_of_the_fireflies,0.000000,0.000000,0.000000,0.006835,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.03504,0.01752,0.000000,0.03504
howls_moving_castle,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.005293,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000
kikis_delivery_service,0.000000,0.000000,0.000000,0.005014,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000
my_neighbor_totoro,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000
nausicaa,0.000000,0.000000,0.161187,0.000000,0.000000,0.007587,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000
only_yesterday,0.000000,0.000000,0.027136,0.000000,0.028582,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.085747,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000
pom_poko,0.000000,0.000000,0.002719,0.002234,0.000000,0.004223,0.005727,0.005727,0.004223,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.011454,0.00000
ponyo,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000


## Save Tables

In [21]:
BOW.to_csv(f'{OUTPUT_DIR}/BOW.csv')
DTM.to_csv(f'{OUTPUT_DIR}/DTM.csv')
TFIDF.to_csv(f'{OUTPUT_DIR}/TFIDF.csv')
TFIDF_L2.to_csv(f'{OUTPUT_DIR}/TFIDF_L2.csv')